In [1]:
# Cell 1: setup
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd

source = [json.loads(line) for line in Path("../data/raw/pcmb_500.jsonl").open()]
traces = [json.loads(line) for line in Path("../data/traces/pcmb_traces.jsonl").open()]

src_df = pd.DataFrame(source)
trc_df = pd.DataFrame(traces)

verified_ids = set(trc_df["id"])
failed = src_df[~src_df["id"].isin(verified_ids)].copy()
print(f"Failed questions: {len(failed)}")
print(failed["subject"].value_counts())

Failed questions: 87
subject
math         31
chemistry    23
physics      18
biology      15
Name: count, dtype: int64


In [2]:
# Cell 2: failure breakdown by subject
# If failures are evenly spread, it's likely parser issues.
# If they're concentrated in one subject, it's a difficulty/domain issue.
src_counts = src_df["subject"].value_counts()
fail_counts = failed["subject"].value_counts().reindex(src_counts.index, fill_value=0)
fail_rate = (fail_counts / src_counts * 100).round(1)
print(pd.DataFrame({
    "sourced": src_counts,
    "failed": fail_counts,
    "fail_rate_%": fail_rate,
}))

           sourced  failed  fail_rate_%
subject                                
physics        125      18         14.4
chemistry      125      23         18.4
biology        125      15         12.0
math           125      31         24.8
